# Civitas — export Model A (optional speed-up)

The local server runs the PyTorch checkpoint in fp16 already, so this notebook is **optional**.
It produces ONNX files that run faster on CPU and use less memory.

## WHAT YOU DO
1. Run notebook 02 first (it leaves `best_model.pt` in your Drive run folder).
2. Set `RUN_NAME` in cell 4 to the same run name, then **Runtime → Run all** (a CPU runtime is fine).
3. Download from Drive `civitas/<RUN_NAME>/export/` into `artifacts/model_a/` on your PC:
   `best_model.pt`, `model_a_int8.onnx`, and the `tokenizer/` folder.
4. Paste back the output of the last cell (agreement between PyTorch and ONNX).

In [ ]:
# 2. Google Drive: everything that must survive a disconnect lives here
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/civitas"
import os
os.makedirs(PROJECT, exist_ok=True)
print("project folder:", PROJECT, "->", sorted(os.listdir(PROJECT)))

In [ ]:
# 3. Get the code: from GitHub (fill REPO_URL) OR from civitas_code.zip in the Drive folder
REPO_URL = ""  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
import os, subprocess, zipfile
CODE = "/content/civitas"
if REPO_URL:
    if os.path.exists(f"{CODE}/.git"):
        subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, CODE], check=True)
else:
    # made on your PC with `make package-code` (or any zip of the repo folder)
    zpath = f"{PROJECT}/civitas_code.zip"
    assert os.path.exists(zpath), f"Set REPO_URL above, or upload civitas_code.zip to {PROJECT}"
    zipfile.ZipFile(zpath).extractall(CODE)
    inner = [d for d in os.listdir(CODE) if os.path.exists(f"{CODE}/{d}/train.py")]
    if not os.path.exists(f"{CODE}/train.py") and inner:  # zip had a top-level folder
        CODE = f"{CODE}/{inner[0]}"
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout
      or "code from zip")

In [ ]:
!pip -q install "transformers>=4.40" onnx onnxruntime pyarrow sentencepiece

In [ ]:
# 4. Load the trained checkpoint
RUN_NAME = "model_a_v2"  #@param {type:"string"}
import os, shutil, torch
OUT = f"{PROJECT}/{RUN_NAME}"
EXPORT = f"{OUT}/export"
os.makedirs(EXPORT, exist_ok=True)
import sys; sys.path.insert(0, CODE)
from infer import load_model
m = load_model(f"{OUT}/best_model.pt", "cpu", half=False)
m.tokenizer.save_pretrained(f"{EXPORT}/tokenizer")
shutil.copy(f"{OUT}/best_model.pt", f"{EXPORT}/best_model.pt")
print("threshold", m.threshold)

In [ ]:
# 5. Export fp32 ONNX (4 outputs: hate, target, severity, rationale logits), then int8
from onnxruntime.quantization import QuantType, quantize_dynamic

class Wrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids, attention_mask):
        o = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return o["hate_logits"], o["target_logits"], o["severity_logits"], o["rationale_logits"]

dummy = m.tokenizer(["export example"], return_tensors="pt")
fp32 = f"{EXPORT}/model_a_fp32.onnx"
torch.onnx.export(
    Wrapper(m.model).eval(), (dummy["input_ids"], dummy["attention_mask"]), fp32,
    input_names=["input_ids", "attention_mask"],
    output_names=["hate_logits", "target_logits", "severity_logits", "rationale_logits"],
    dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"},
                  "rationale_logits": {0: "batch", 1: "seq"}, "hate_logits": {0: "batch"},
                  "target_logits": {0: "batch"}, "severity_logits": {0: "batch"}},
    opset_version=17, dynamo=False)
int8 = f"{EXPORT}/model_a_int8.onnx"
quantize_dynamic(fp32, int8, weight_type=QuantType.QInt8)
for f in (fp32, int8):
    print(os.path.basename(f), f"{os.path.getsize(f) / 1e6:.0f} MB")

In [ ]:
# 6. Agreement check on 200 val rows: PyTorch vs ONNX int8. Paste this output back.
import numpy as np, onnxruntime as ort, pandas as pd, zipfile
zipfile.ZipFile(f"{PROJECT}/civitas_data_v2.zip").extract("val.parquet", "/content")
texts = pd.read_parquet("/content/val.parquet")["text"].astype(str).sample(200, random_state=0).tolist()
from infer import score_batch
from text_norm import normalize_code_mixed
pt = np.array([r["hate_prob"] for r in score_batch(m, texts)])
sess = ort.InferenceSession(int8, providers=["CPUExecutionProvider"])
onx = []
for t in texts:
    enc = m.tokenizer(normalize_code_mixed(t)[0], truncation=True, max_length=128, return_tensors="np")
    logits = sess.run(["hate_logits"], {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]})[0]
    e = np.exp(logits - logits.max(-1, keepdims=True))
    onx.append((e / e.sum(-1, keepdims=True))[0, 1])
onx = np.array(onx)
agree = ((pt >= m.threshold) == (onx >= m.threshold)).mean()
print(f"max |p_pytorch - p_onnx_int8| = {np.abs(pt - onx).max():.4f}")
print(f"mean |diff| = {np.abs(pt - onx).mean():.4f}   label agreement = {agree:.1%}")